# International metric verification audit
Snapshot: 12 October 2026. Grain: one academic per source ID. Checks reconcile source, provider snapshots and app CSVs; they do not prove exhaustive lifetime publication coverage. Identity evidence and rejected matches are in the linked JSON audit.
Unknown metrics stay blank. OpenAlex totals are provider-specific and are not interchangeable with Google Scholar.

In [1]:
from pathlib import Path
import csv, json
from collections import Counter
root = Path.cwd()
if not (root / "Data files/international").exists():
    root = root.parents[1]  # running from scripts/baselines
source = root / "Data files/international"
def read(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))
audit = json.loads((source / "_metric_verification_2026-10-12.json").read_text())["records"]
assert len(audit) == 110 and len({r["academic_id"] for r in audit}) == 110
rows = [r for p in source.glob("*/universities/*/faculties/*/departments/*/academics.csv") for r in read(p)]
by_id = {r["academic_id"]: r for r in rows}
assert len(by_id) == len(rows) == 1740
published = {r["id"]: r for r in read(root / "public/data/international/academics.csv")}
fields = {"total_publications": "publications", "total_citations": "citations", "h_index": "hIndex", "citations_2021_2025": "recentCitations"}
for v in audit:
    r, public = by_id[v["academic_id"]], published[v["academic_id"]]
    assert r["research_lookup_status"] == v["status"]
    for source_field, public_field in fields.items():
        expected = str(v["metrics"][source_field]) if v["matched"] else ""
        assert r[source_field] == public[public_field] == expected
    if v["matched"]:
        assert v["evidence"] and v["metrics_source_url"] and v["retrieved_at"] == "2026-10-12"
        a = v["provider_snapshot"]
        assert v["metrics"]["total_publications"] == a["works_count"]
        assert v["metrics"]["total_citations"] == a["cited_by_count"]
        assert v["metrics"]["h_index"] == a["summary_stats"]["h_index"]
        assert v["metrics"]["citations_2021_2025"] == sum(y["cited_by_count"] for y in a["counts_by_year"] if 2021 <= y["year"] <= 2025)
        assert 0 <= v["metrics"]["h_index"] <= v["metrics"]["total_publications"]
        assert 0 <= v["metrics"]["citations_2021_2025"] <= v["metrics"]["total_citations"]
assert sum(v["matched"] for v in audit) == 102
assert sum(r["includedInBenchmarks"] == "false" for r in published.values()) == 1
assert sum(r["includedInBenchmarks"] != "false" for r in published.values()) == 1739
pubs = read(source / "publications.csv")
assert len(pubs) == 8496 and all(p["academic_id"] in by_id for p in pubs)
assert len(read(root / "public/data/international/publications.csv")) == len(pubs)
print("110 checked; 102 corroborated metric snapshots; 8 unresolved with blank values.")
print("1,740 source records; 1,739 benchmark-eligible; 8,496 publication attributions.")


110 checked; 102 corroborated metric snapshots; 8 unresolved with blank values.
1,740 source records; 1,739 benchmark-eligible; 8,496 publication attributions.
